In [27]:
## Read dataset
import pandas as pd

## Row : 891 개
dtf = pd.read_csv('http://bit.ly/kaggletrain')
dtf.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [28]:
## Create DB
import sqlite3

dtf.to_sql(index=False, name="titanic", con=sqlite3.connect("database.db"), if_exists="replace")

891

In [29]:
## Connect DB
from langchain_community.utilities.sql_database import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///database.db")

In [30]:
from langchain_community.tools.sql_database.tool import ListSQLDatabaseTool

def get_tables() -> str:
    return ListSQLDatabaseTool(db=db).invoke("")

tool_get_tables = {'type':'function', 'function':{
    'name': 'get_tables',
    'description': 'Returns the name of the tables in the database.',
    'parameters': {'type': 'object',
                   'required': [],
                   'properties': {}
                  }}}

## test
get_tables()

'titanic'

In [31]:
from langchain_community.tools.sql_database.tool import InfoSQLDatabaseTool

def get_schema(tables: str) -> str:
    tool = InfoSQLDatabaseTool(db=db)
    return tool.invoke(tables)

tool_get_schema = {'type':'function', 'function':{
    'name': 'get_schema',
    'description': 'Returns the name of the columns in the table.',
    'parameters': {'type': 'object',
                   'required': ['tables'],
                   'properties': {'tables': {'type':'str', 'description':'table name. Example Input: table1, table2, table3'}}
                  }}}

## test
get_schema(tables='titanic')

'\nCREATE TABLE titanic (\n\t"PassengerId" INTEGER, \n\t"Survived" INTEGER, \n\t"Pclass" INTEGER, \n\t"Name" TEXT, \n\t"Sex" TEXT, \n\t"Age" REAL, \n\t"SibSp" INTEGER, \n\t"Parch" INTEGER, \n\t"Ticket" TEXT, \n\t"Fare" REAL, \n\t"Cabin" TEXT, \n\t"Embarked" TEXT\n)\n\n/*\n3 rows from titanic table:\nPassengerId\tSurvived\tPclass\tName\tSex\tAge\tSibSp\tParch\tTicket\tFare\tCabin\tEmbarked\n1\t0\t3\tBraund, Mr. Owen Harris\tmale\t22.0\t1\t0\tA/5 21171\t7.25\tNone\tS\n2\t1\t1\tCumings, Mrs. John Bradley (Florence Briggs Thayer)\tfemale\t38.0\t1\t0\tPC 17599\t71.2833\tC85\tC\n3\t1\t3\tHeikkinen, Miss. Laina\tfemale\t26.0\t0\t0\tSTON/O2. 3101282\t7.925\tNone\tS\n*/'

In [32]:
prompt_junior = '''
[GOAL] You are a data engineer who builds efficient SQL queries to get data from the database.

[RETURN] You must return a final SQL query based on user's instructions.

[WARNINGS] Use your tools only once.

[CONTEXT] In order to generate the perfect SQL query, you need to know the name of the table and the schema.
First ALWAYS use the tool 'get_tables' to find the name of the table.
Then, you MUST use the tool 'get_schema' to get the columns in the table.
Finally, based on the information you got, generate an SQL query to answer user question.
'''

In [33]:
#from langchain_community.tools.sql_database.tool import QuerySQLCheckerTool
#def sql_check(sql: str) -> str:
#    return QuerySQLCheckerTool(db=db, llm=llm).invoke({"query":sql})

import ollama
llm = "qwen2.5"

def sql_check(sql: str) -> str:
    p = f'''Double check if the SQL query is correct: {sql}. You MUST just SQL code without comments'''
    res = ollama.generate(model=llm, prompt=p)["response"]
    return res.replace('sql','').replace('```','').replace('\n',' ').strip()

tool_sql_check = {'type':'function', 'function':{
    'name': 'sql_check',
    'description': 'Before executing a query, always review the SQL query and correct the code if necessary',
    'parameters': {'type': 'object',
                   'required': ['sql'],
                   'properties': {'sql': {'type':'str', 'description':'SQL code'}}
                  }}}

## test
sql_check(sql='SELECT * FROM titanic TOP 3')

'SELECT * FROM titanic LIMIT 3;'

In [34]:
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool

def sql_exec(sql: str) -> str:
    return QuerySQLDatabaseTool(db=db).invoke(sql)

tool_sql_exec = {'type':'function', 'function':{
    'name': 'sql_exec',
    'description': 'Execute a SQL query',
    'parameters': {'type': 'object',
                   'required': ['sql'],
                   'properties': {'sql': {'type':'str', 'description':'SQL code'}}
                  }}}

## test
sql_exec(sql='SELECT * FROM titanic LIMIT 3')

"[(1, 0, 3, 'Braund, Mr. Owen Harris', 'male', 22.0, 1, 0, 'A/5 21171', 7.25, None, 'S'), (2, 1, 1, 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)', 'female', 38.0, 1, 0, 'PC 17599', 71.2833, 'C85', 'C'), (3, 1, 3, 'Heikkinen, Miss. Laina', 'female', 26.0, 0, 0, 'STON/O2. 3101282', 7.925, None, 'S')]"

In [35]:
prompt_senior = '''[GOAL] You are a senior data engineer who reviews and execute the SQL queries written by others.

[RETURN] You must return data from the database.

[WARNINGS] Use your tools only once.

[CONTEXT] ALWAYS check the SQL code before executing on the database.
First ALWAYS use the tool 'sql_check' to review the query.
The output of this tool is the correct SQL query.
You MUST use ONLY the correct SQL query when you use the tool 'sql_exec'.
'''

In [36]:
def invoke_agent(agent:str, instructions:str) -> str:
    return agent+" - "+instructions if agent in ['junior','senior'] else f"Agent '{agent}' Not Found"

tool_invoke_agent = {'type':'function', 'function':{
    'name': 'invoke_agent',
    'description': 'Invoke another Agent to work for you.',
    'parameters': {'type': 'object',
                   'required': ['agent', 'instructions'],
                   'properties': {
                       'agent': {'type':'str', 'description':'the Agent name, one of "junior" or "senior".'},
                       'instructions': {'type':'str', 'description':'detailed instructions for the Agent.'}
                   }
                  }}}

## test
invoke_agent(agent="intern", instructions="build a query")

"Agent 'intern' Not Found"

In [37]:
prompt_lead = '''
[GOAL] You are a tech lead.
You have a team with one junior data engineer called 'junior', and one senior data engineer called 'senior'.

[RETURN] You must return data from the database based on user's requests.

[WARNINGS] You are the only one that talks to the user and gets the requests from the user.
The 'junior' data engineer only builds queries.
The 'senior' data engineer checks the queries and execute them.

[CONTEXT] First ALWAYS ask the users what they want.
Then, you MUST use the tool, 'invoke_agent' to pass the instructions to the 'junior' for building the query.
Finally, you MUST use the tool, 'invoke_agent' to pass the instructions to the 'senior' for retrieving the data from the database.
'''

In [38]:
dic_tools = {'get_tables':get_tables,
             'get_schema':get_schema,
             'sql_exec':sql_exec,
             'sql_check':sql_check,
             'invoke_agent':invoke_agent}

messages_junior = [{"role":"system", "content":prompt_junior}]
messages_senior = [{"role":"system", "content":prompt_senior}]
messages_lead = [{"role":"system", "content":prompt_lead}]

In [39]:
def use_tool(agent_res:dict, dic_tools:dict) -> dict:
    ## 기본값 설정
    res = agent_res["message"].get("content", "")
    t_name, t_inputs = '', ''
    
    ## 도구 사용 여부 확인 (딕셔너리 키로 확인)
    if "tool_calls" in agent_res["message"]:
        for tool in agent_res["message"]["tool_calls"]:
            t_name, t_inputs = tool["function"]["name"], tool["function"]["arguments"]
            
            if f := dic_tools.get(t_name):
                ### calling tool
                print('🔧 >', f"\x1b[1;31m{t_name} -> Inputs: {t_inputs}\x1b[0m")
                ### tool output
                t_output = f(**tool["function"]["arguments"])
                print(t_output)
                ### final res
                res = t_output
            else:
                print('🤬 >', f"\x1b[1;31m{t_name} -> NotFound\x1b[0m")

    return {'res':res, 'tool_used':t_name, 'inputs_used':t_inputs}

In [40]:
while True:
    ## user input
    q = input('🙂 >')
    if q == "quit":
        break
    messages_lead.append( {"role":"user", "content":q} )

    ## Lead Agent
    agent_res = ollama.chat(model=llm, messages=messages_lead, tools=[tool_invoke_agent])
    dic_res = use_tool(agent_res, dic_tools)
    res, tool_used, inputs_used = dic_res["res"], dic_res["tool_used"], dic_res["inputs_used"]
    agent_invoked = res.split("-")[0].strip() if len(res.split("-")) > 1 else ''
    instructions = res.split("-")[1].strip() if len(res.split("-")) > 1 else ''

    ## Invoke Junior Agent
    if agent_invoked == "junior":
        print("😎 >", f"\x1b[1;32mReceived instructions: {instructions}\x1b[0m")
        messages_junior.append( {"role":"user", "content":instructions} )

        ### use the tools
        available_tools = {"get_tables":tool_get_tables, "get_schema":tool_get_schema}
        context = ''

        while available_tools:
            agent_res = ollama.chat(model=llm, messages=messages_junior,
                                    tools=[v for v in available_tools.values()])
            dic_res = use_tool(agent_res, dic_tools)
            res, tool_used, inputs_used = dic_res["res"], dic_res["tool_used"], dic_res["inputs_used"]

            if tool_used:
                available_tools.pop(tool_used)

            context = context + f"\nTool used: {tool_used}. Output: {res}"  # -> add tool usage context
            messages_junior.append( {"role":"user", "content":context} )

        ### response
        agent_res = ollama.chat(model=llm, messages=messages_junior)
        dic_res = use_tool(agent_res, dic_tools)
        res = dic_res["res"]
        print("😎 >", f"\x1b[1;32m{res}\x1b[0m")
        messages_junior.append( {"role":"assistant", "content":res} )

        ### Update Lead Agent
        context = "Junior already wrote this query: "+res+ "\nNow invoke the Senior to review and execute the code."
        print("👩‍💼 >", f"\x1b[1;30m{context}\x1b[0m")
        messages_lead.append( {"role":"user", "content":context} )
        agent_res = ollama.chat(model=llm, messages=messages_lead, tools=[tool_invoke_agent])
        dic_res = use_tool(agent_res, dic_tools)
        res, tool_used, inputs_used = dic_res["res"], dic_res["tool_used"], dic_res["inputs_used"]
        agent_invoked = res.split("-")[0].strip() if len(res.split("-")) > 1 else ''
        instructions = res.split("-")[1].strip() if len(res.split("-")) > 1 else ''

    ## Invoke Senior Agent
    if agent_invoked == "senior":
        print("🧓 >", f"\x1b[1;32mReceived instructions: {instructions}\x1b[0m")
        messages_senior.append( {"role":"user", "content":instructions} )

        ### use the tools
        available_tools = {"sql_check":tool_sql_check, "sql_exec":tool_sql_exec}
        context = ''

        while available_tools:
            agent_res = ollama.chat(model=llm, messages=messages_senior,
                                    tools=[v for v in available_tools.values()])
            dic_res = use_tool(agent_res, dic_tools)
            res, tool_used, inputs_used = dic_res["res"], dic_res["tool_used"], dic_res["inputs_used"]

            if tool_used:
                available_tools.pop(tool_used)

            context = context + f"\nTool used: {tool_used}. Output: {res}"  # -> add tool usage context
            messages_senior.append( {"role":"user", "content":context} )

        ### response
        print("🧓 >", f"\x1b[1;32m{res}\x1b[0m")
        messages_senior.append( {"role":"assistant", "content":res} )

        ### Update Lead Agent
        context = "Senior agent returned this output: "+res
        print("👩‍💼 >", f"\x1b[1;30m{context}\x1b[0m")
        messages_lead.append( {"role":"user", "content":context} )

    ## Lead Agent final response
    print("👩‍💼 >", f"\x1b[1;30m{res}\x1b[0m")
    messages_lead.append( {"role":"assistant", "content":res} )

🙂 > How many women are in the db?


👩‍💼 > To provide you with accurate data, I need to know which table contains this information. Could you please specify the table name that holds the gender attribute?


🙂 > titanic


🔧 > invoke_agent -> Inputs: {'agent': 'junior', 'instructions': 'Build a query to count the number of women in the titanic table.'}
junior - Build a query to count the number of women in the titanic table.
😎 > Received instructions: Build a query to count the number of women in the titanic table.
🔧 > get_tables -> Inputs: {}
titanic
🔧 > get_schema -> Inputs: {'tables': 'titanic'}

CREATE TABLE titanic (
	"PassengerId" INTEGER, 
	"Survived" INTEGER, 
	"Pclass" INTEGER, 
	"Name" TEXT, 
	"Sex" TEXT, 
	"Age" REAL, 
	"SibSp" INTEGER, 
	"Parch" INTEGER, 
	"Ticket" TEXT, 
	"Fare" REAL, 
	"Cabin" TEXT, 
	"Embarked" TEXT
)

/*
3 rows from titanic table:
PassengerId	Survived	Pclass	Name	Sex	Age	SibSp	Parch	Ticket	Fare	Cabin	Embarked
1	0	3	Braund, Mr. Owen Harris	male	22.0	1	0	A/5 21171	7.25	None	S
2	1	1	Cumings, Mrs. John Bradley (Florence Briggs Thayer)	female	38.0	1	0	PC 17599	71.2833	C85	C
3	1	3	Heikkinen, Miss. Laina	female	26.0	0	0	STON/O2. 3101282	7.925	None	S
*/
😎 > Based on the schema pr

🙂 > quit
